In [ ]:
# NOTEBOOK 02 — REDUCCIÓN Y EXPLORATORIO INICIAL DE CONTAMINANTES
# preparación de los datasets de contaminación atmosférica

"""
en este notebook realizo la carga, reducción y exploratorio inicial de los datasets
de contaminantes atmosféricos (PM2.5, PM10, NO2, O3, SO2, CO y C6H6)

los archivos originales contienen millones de registros horarios, por lo que se aplican
técnicas de carga por partes (row groups), filtrado temporal al rango 2011-2019 y
agregación diaria para obtener datasets manejables, coherentes y comparables con los
datasets de salud pública

el objetivo de este notebook es dejar todos los contaminantes en un formato limpio,
homogéneo y reducido, listo para integrarse posteriormente con las estaciones y los
datasets de contexto en el Notebook 04.
"""

In [2]:
# importaciones de librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
'''esto lo meto porque es una configuración visual de pandas para que no me oculte columnas
cuando haga df.head() o df.info():'''
pd.set_option('display.max_columns', None)

In [ ]:
# estoy teniendo problemas al cargar los datos de los contaminantes porque hay millones de filas,
# así que voy a reducirlos, primero poniendo en común los años que necesito 

In [4]:
# cargo dataset de salud:

df_hospi = pd.read_csv("../data/hospitalizations_spain.csv")
df_mortality = pd.read_csv("../data/mortality_respiratory_spain.csv")
df_population = pd.read_csv("../data/population_spain.csv")
df_stations = pd.read_csv("../data/stations_metadata_spain.csv")

In [5]:
# primero miro a ver cómo se llaman las columnas de fecha

print("Hospitalizaciones:", df_hospi.columns)
print("Mortalidad:", df_mortality.columns)
print("Población:", df_population.columns)
print("Estaciones:", df_stations.columns)


Hospitalizaciones: Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency',
       'age', 'Age class', 'indic_he', 'Health indicator', 'unit',
       'Unit of measure', 'sex', 'Sex', 'icd10',
       'International Statistical Classification of Diseases and Related Health Problems (ICD-10)',
       'geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD', 'Time',
       'OBS_VALUE', 'Observation value', 'OBS_FLAG',
       'Observation status (Flag) V2 structure', 'CONF_STATUS',
       'Confidentiality status (flag)'],
      dtype='str')
Mortalidad: Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency',
       'unit', 'Unit of measure', 'sex', 'Sex', 'age', 'Age class', 'icd10',
       'International Statistical Classification of Diseases and Related Health Problems (ICD-10)',
       'resid', 'Place of residence', 'geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD', 'Time', 'OBS_VALUE', 'Observation value', 'OBS_FLAG',
       'Obs

In [6]:
df_hospi["TIME_PERIOD"].head(20)


0     2000
1     2001
2     2002
3     2003
4     2004
5     2005
6     2006
7     2007
8     2008
9     2009
10    2010
11    2011
12    2012
13    2013
14    2014
15    2015
16    2016
17    2017
18    2018
19    2019
Name: TIME_PERIOD, dtype: int64

In [7]:
df_mortality["TIME_PERIOD"].head(20)



0     2011
1     2012
2     2013
3     2014
4     2015
5     2016
6     2017
7     2018
8     2019
9     2020
10    2021
11    2022
12    2023
13    2024
14    2011
15    2012
16    2013
17    2014
18    2015
19    2016
Name: TIME_PERIOD, dtype: int64

In [8]:
df_population["TIME_PERIOD"].head(20)

0     2000
1     2001
2     2002
3     2003
4     2004
5     2005
6     2006
7     2007
8     2008
9     2009
10    2010
11    2011
12    2012
13    2013
14    2014
15    2015
16    2016
17    2017
18    2018
19    2019
Name: TIME_PERIOD, dtype: int64

In [9]:
# RANGO TEMPORAL DEL PROYECTO

year_start = 2011
year_end = 2019

'''
Hospitalizaciones: 2000-2019
Mortalidad: 2011-2024
Población: 2000-2019
El rango común es: 2011-2019
Este rango:

es consistente

permite unir todos los datasets

evita huecos temporales

es perfecto para análisis de salud pública

reduce muchísimo el tamaño de los contaminantes
'''

'\nHospitalizaciones: 2000-2019\nMortalidad: 2011-2024\nPoblación: 2000-2019\nEl rango común es: 2011-2019\nEste rango:\n\nes consistente\n\npermite unir todos los datasets\n\nevita huecos temporales\n\nes perfecto para análisis de salud pública\n\nreduce muchísimo el tamaño de los contaminantes\n'

In [ ]:
# función para cargar Parquet grandes por partes

import pyarrow.parquet as pq

def load_big_parquet(path, columns=None):
    '''
    carga un Parquet muy grande por partes (row groups) para no saturar la RAM
    - lee el archivo Parquet con pyarrow
    - recorre cada row group y lo convierte a pandas
    - guarda cada fragmento en una lista
    - concatena todos los fragmentos en un único DataFrame
    devuelve el DataFrame completo cargado de forma segura
    '''
    pf = pq.ParquetFile(path)
    dfs = []
    for i in range(pf.num_row_groups):
        print(f"Cargando row group {i+1}/{pf.num_row_groups} de {path}...")
        chunk = pf.read_row_group(i, columns=columns).to_pandas()
        dfs.append(chunk)
    return pd.concat(dfs, ignore_index=True)

In [11]:
# voy a ver cómo se llaman las columnas de los datos de los contaminantes

pf = pq.ParquetFile("../data/pollutants/processed/pm25.parquet")
[col.name for col in pf.schema]


['Samplingpoint',
 'Pollutant',
 'Start',
 'End',
 'Value',
 'Unit',
 'AggType',
 'Validity',
 'Verification',
 'ResultTime',
 'DataCapture',
 'FkObservationLog']

In [ ]:
'''
columnas principales del Parquet:
    Start -> fecha/hora de inicio de la medición (equivale a DatetimeBegin)
    Samplingpoint -> código de la estación (equivale a AirQualityStation)
    Value -> concentración medida del contaminante (equivale a Concentration)
'''

In [ ]:
'''
ya cargada:
import os
'''
os.listdir("../data/pollutants/processed")

['c6h6.parquet',
 'co.parquet',
 'no2.parquet',
 'o3.parquet',
 'pm10.parquet',
 'pm25.parquet',
 'so2.parquet']

In [12]:
# ahora tengo que reducir contaminantes

# crear carpeta para versiones reducidas
os.makedirs("../data/pollutants/processed_small", exist_ok=True)

# rango temporal común entre datasets de salud
year_start = 2011
year_end = 2019

contaminantes = ["pm25", "pm10", "no2", "o3", "so2", "co", "c6h6"]

for cont in contaminantes:
    print(f"\n=== Procesando {cont.upper()} ===")
    
    # cargo por partes solo las columnas necesarias
    df = load_big_parquet(
        f"../data/pollutants/processed/{cont}.parquet",
        columns=["Start", "Samplingpoint", "Value"])
    # convierto fecha
    df["Start"] = pd.to_datetime(df["Start"], errors="coerce")
    # convierto Value a numérico
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    # filtro por años útiles
    df = df[df["Start"].dt.year.between(year_start, year_end)]
    # agrego a nivel diario
    df["date"] = df["Start"].dt.date
    df_daily = df.groupby(["date", "Samplingpoint"])["Value"].mean().reset_index()
    # guardar versión reducida
    df_daily.to_parquet(f"../data/pollutants/processed_small/{cont}_daily.parquet")

    print(f"{cont.upper()} reducido y guardado.")


=== Procesando PM25 ===
Cargando row group 1/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 2/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 3/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 4/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 5/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 6/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 7/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 8/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 9/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 10/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 11/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 12/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 13/17 de ../data/pollutants/processed/pm25.parquet...
Cargando row group 14/17 de ../data

KeyboardInterrupt: 

In [14]:
os.listdir("../data/pollutants/processed_small")

['c6h6_daily.parquet',
 'co_daily.parquet',
 'no2_daily.parquet',
 'o3_daily.parquet',
 'pm10_daily.parquet',
 'pm25_daily.parquet',
 'so2_daily.parquet']

In [16]:
# una vez los dataset están reducidos, los cargo

df_pm25 = pd.read_parquet("../data/pollutants/processed_small/pm25_daily.parquet")
df_pm10 = pd.read_parquet("../data/pollutants/processed_small/pm10_daily.parquet")
df_no2  = pd.read_parquet("../data/pollutants/processed_small/no2_daily.parquet")
df_o3   = pd.read_parquet("../data/pollutants/processed_small/o3_daily.parquet")
df_so2  = pd.read_parquet("../data/pollutants/processed_small/so2_daily.parquet")
df_co   = pd.read_parquet("../data/pollutants/processed_small/co_daily.parquet")
df_c6h6 = pd.read_parquet("../data/pollutants/processed_small/c6h6_daily.parquet")

print("Todos los contaminantes cargados correctamente (versión reducida).")


Todos los contaminantes cargados correctamente (versión reducida).


In [14]:
df_pm25.columns

Index(['date', 'Samplingpoint', 'Value'], dtype='str')

In [15]:
df_pm25

,date,Samplingpoint,Value
0,2011-01-01,ES/SP_02003001_9_49,16.447619
1,2011-01-01,ES/SP_09059006_9_49,10.956522
2,2011-01-01,ES/SP_28007004_9_49,19.739130
3,2011-01-01,ES/SP_28148004_9_49,14.409091
4,2011-01-02,ES/SP_02003001_9_49,15.869565
...,...,...,...
319941,2019-12-31,ES/SP_48020003_9_49,10.833333
319942,2019-12-31,ES/SP_48027001_9_49,14.708333
319943,2019-12-31,ES/SP_48096001_9_49,15.500000
319944,2019-12-31,ES/SP_48902006_9_49,12.625000


In [ ]:
# ahore toca el exploratorio inicial para cada contaminante: número de columnas, filas, tipos de datos, primeros registros,
# missings, duplicados, valores únicos..

In [17]:
# PM2.5

name_pm25 = "PM25"

print("="*80)
print(f"EXPLORATORIO INICIAL — {name_pm25}")
print("="*80)

# shape
print("\nShape (filas, columnas):")
print(df_pm25.shape)
# info
print("\nInfo:")
print(df_pm25.info())
# primeras filas
print("\nPrimeras filas:")
print(df_pm25.head())
# tipos de datos
print("\nTipos de datos:")
print(df_pm25.dtypes)
# missings
print("\nPorcentaje de missings por columna:")
print((df_pm25.isna().mean() * 100).sort_values(ascending=False))
# duplicados
print("\nNúmero de duplicados:")
print(df_pm25.duplicated().sum())
# valores únicos por columna
print("\nNúmero de valores únicos por columna:")
print(df_pm25.nunique())


EXPLORATORIO INICIAL — PM25

Shape (filas, columnas):
(319946, 3)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 319946 entries, 0 to 319945
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   date           319946 non-null  object 
 1   Samplingpoint  319946 non-null  str    
 2   Value          319946 non-null  float64
dtypes: float64(1), object(1), str(1)
memory usage: 13.1+ MB
None

Primeras filas:
         date        Samplingpoint      Value
0  2011-01-01  ES/SP_02003001_9_49  16.447619
1  2011-01-01  ES/SP_09059006_9_49  10.956522
2  2011-01-01  ES/SP_28007004_9_49  19.739130
3  2011-01-01  ES/SP_28148004_9_49  14.409091
4  2011-01-02  ES/SP_02003001_9_49  15.869565

Tipos de datos:
date              object
Samplingpoint        str
Value            float64
dtype: object

Porcentaje de missings por columna:
date             0.0
Samplingpoint    0.0
Value            0.0
dtype: float64

Número de duplicados:


In [ ]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE PM2.5

Shape (319.946, 3): el dataset contiene casi 320.000 registros diarios entre 2011-2019 cada fila representa
    una combinación única de fecha y estación
    es un dataset temporal y espacial, no agregado

Tipos: las columnas tienen los tipos esperados: date como object (convertible a datetime), Samplingpoint
    como string y Value como float
    no hay columnas adicionales ni metadatos innecesarios

Primeras filas: los valores de PM2.5 entre 10-20 µg/m³ son realistas para invierno en España
    los códigos de estación siguen el formato estándar EEA (ES/SP_XXXXXXXX_X_XX).

Missings: 0% en todas las columnas, la reducción diaria eliminó cualquier hueco horario

Duplicados: 0 duplicados, cada combinación date + Samplingpoint es única

Valores únicos: 3287 fechas, 182 estaciones y 23.952 valores distintos
    
'''

'\n-Interpretación de (319946, 3):\n\n    Tienes 319.946 registros diarios por estación entre 2011–2019.\n    Es un dataset grande pero manejable.\n    Las 3 columnas (date, Samplingpoint, Value) son exactamente las esperadas tras la reducción.\n\n\n-Interpretación de\n                    date | object\n                    Samplingpoint | str\n                    Value | float64\n\n    date está como object, pero no pasa nada porque ya la convertimos antes.\n    Si quieres, podemos convertirla a datetime ahora para el EDA temporal.\n\n    Samplingpoint es string → correcto.\n\n    Value es float64 → perfecto para análisis numérico.\n\n    No hay columnas raras, ni tipos incorrectos.\n\n\n-Primeras filas, interpretación:\n\n    Los valores tienen sentido: PM2.5 entre 10-20 µg/m³ en enero → totalmente realista.\n\n    Las estaciones están en formato EEA estándar: ES/SP_28007004_9_49.\n\n    No hay valores negativos ni absurdos.\n\n-Missings:\n\n\n'

In [18]:
name_pm10 = "PM10"

print("="*80)
print(f"EXPLORATORIO INICIAL — {name_pm10}")
print("="*80)

print("\nShape (filas, columnas):")
print(df_pm10.shape)
print("\nInfo:")
print(df_pm10.info())
print("\nPrimeras filas:")
print(df_pm10.head())
print("\nTipos de datos:")
print(df_pm10.dtypes)
print("\nPorcentaje de missings por columna:")
print((df_pm10.isna().mean() * 100).sort_values(ascending=False))
print("\nNúmero de duplicados:")
print(df_pm10.duplicated().sum())
print("\nNúmero de valores únicos por columna:")
print(df_pm10.nunique())

EXPLORATORIO INICIAL — PM10

Shape (filas, columnas):
(701961, 3)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 701961 entries, 0 to 701960
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   date           701961 non-null  object 
 1   Samplingpoint  701961 non-null  str    
 2   Value          701961 non-null  float64
dtypes: float64(1), object(1), str(1)
memory usage: 29.5+ MB
None

Primeras filas:
         date         Samplingpoint      Value
0  2013-01-01  ES/SP_01022001_10_47   6.608696
1  2013-01-01  ES/SP_01036004_10_47   0.000000
2  2013-01-01  ES/SP_01055001_10_47  10.782609
3  2013-01-01  ES/SP_02003001_10_49  24.886957
4  2013-01-01  ES/SP_03014009_10_46   4.434783

Tipos de datos:
date              object
Samplingpoint        str
Value            float64
dtype: object

Porcentaje de missings por columna:
date             0.0
Samplingpoint    0.0
Value            0.0
dtype: float64

Número de duplic

In [ ]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE PM10

Shape (701.961, 3): dataset grande, con más de 700.000 registros diarios
    PM10 se mide en más estaciones que PM2.5, por eso el tamaño es mayor

Tipos: date como object, Samplingpoint como string y Value como float
    estructura correcta y homogénea

Primeras filas: Los valores 0-25 µg/m³ son coherentes con niveles típicos de PM10 en España
    los códigos de estación siguen el formato estándar

Missings: 0% en todas las columnas

Duplicados: Ninguno, la agregación diaria está correctamente aplicada

Valores únicos: 2556 fechas, 375 estaciones y 63.480 valores distintos
    es dataset robusto para análisis temporal, espacial y comparaciones entre estaciones

'''

In [19]:
name_no2 = "NO2"

print("="*80)
print(f"EXPLORATORIO INICIAL — {name_no2}")
print("="*80)

print("\nShape (filas, columnas):")
print(df_no2.shape)
print("\nInfo:")
print(df_no2.info())
print("\nPrimeras filas:")
print(df_no2.head())
print("\nTipos de datos:")
print(df_no2.dtypes)
print("\nPorcentaje de missings por columna:")
print((df_no2.isna().mean() * 100).sort_values(ascending=False))
print("\nNúmero de duplicados:")
print(df_no2.duplicated().sum())
print("\nNúmero de valores únicos por columna:")
print(df_no2.nunique())


EXPLORATORIO INICIAL — NO2

Shape (filas, columnas):
(1216735, 3)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 1216735 entries, 0 to 1216734
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   date           1216735 non-null  object 
 1   Samplingpoint  1216735 non-null  str    
 2   Value          1216735 non-null  float64
dtypes: float64(1), object(1), str(1)
memory usage: 48.7+ MB
None

Primeras filas:
         date       Samplingpoint      Value
0  2013-01-01  ES/SP_01022001_8_8   7.869565
1  2013-01-01  ES/SP_01036004_8_8  16.826087
2  2013-01-01  ES/SP_01055001_8_8   5.130435
3  2013-01-01  ES/SP_01059009_8_8  26.478261
4  2013-01-01  ES/SP_01059012_8_8  24.652174

Tipos de datos:
date              object
Samplingpoint        str
Value            float64
dtype: object

Porcentaje de missings por columna:
date             0.0
Samplingpoint    0.0
Value            0.0
dtype: float64

Número de duplicados:

In [ ]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE NO2

Shape (1.216.735, 3): es uno de los datasets más grandes, lo cual es normal porque NO2 se monitoriza
    en muchas estaciones urbanas y de tráfico

Tipos: date como object, Samplingpoint como string y Value como float

Primeras filas: Los valores 5-25 µg/m³ son realistas para invierno
    los códigos de estación siguen el formato EEA

Missings: 0% en todas las columnas

Duplicados: ninguno, cada registro diario es único

Valores únicos: 2556 fechas, 549 estaciones y 72.649 valores distintos
    dataset ideal para estudiar patrones urbanos, tráfico y variabilidad espacial
'''

In [20]:
name_o3 = "O3"

print("="*80)
print(f"EXPLORATORIO INICIAL — {name_o3}")
print("="*80)

print("\nShape (filas, columnas):")
print(df_o3.shape)
print("\nInfo:")
print(df_o3.info())
print("\nPrimeras filas:")
print(df_o3.head())
print("\nTipos de datos:")
print(df_o3.dtypes)
print("\nPorcentaje de missings por columna:")
print((df_o3.isna().mean() * 100).sort_values(ascending=False))
print("\nNúmero de duplicados:")
print(df_o3.duplicated().sum())
print("\nNúmero de valores únicos por columna:")
print(df_o3.nunique())


EXPLORATORIO INICIAL — O3

Shape (filas, columnas):
(1280704, 3)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 1280704 entries, 0 to 1280703
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   date           1280704 non-null  object 
 1   Samplingpoint  1280704 non-null  str    
 2   Value          1280704 non-null  float64
dtypes: float64(1), object(1), str(1)
memory usage: 52.5+ MB
None

Primeras filas:
         date        Samplingpoint      Value
0  2011-01-01  ES/SP_01022001_14_6  38.608696
1  2011-01-01  ES/SP_01036004_14_6  10.173913
2  2011-01-01  ES/SP_01051001_14_6  23.782609
3  2011-01-01  ES/SP_01055001_14_6  41.500000
4  2011-01-01  ES/SP_02003001_14_6  33.416667

Tipos de datos:
date              object
Samplingpoint        str
Value            float64
dtype: object

Porcentaje de missings por columna:
date             0.0
Samplingpoint    0.0
Value            0.0
dtype: float64

Número de duplic

In [ ]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE O3

Shape (1.280.704, 3): es el dataset más grande, coherente con el hecho de que O3 se mide en una red
    muy extensa de estaciones

Tipos: date como object, Samplingpoint como string y Value como float

Primeras filas: Valores entre 10-40 µg/m³, totalmente realistas para invierno
    O3 suele tener picos en verano, lo cual se podrá analizar después

Missings: 0% en todas las columnas

Duplicados: Ninguno

Valores únicos: 3287 fechas, 466 estaciones y 126.404 valores distintos
    dataset perfecto para estudiar estacionalidad y variabilidad regional
'''

In [21]:
name_so2 = "SO2"

print("="*80)
print(f"EXPLORATORIO INICIAL — {name_so2}")
print("="*80)

print("\nShape (filas, columnas):")
print(df_so2.shape)
print("\nInfo:")
print(df_so2.info())
print("\nPrimeras filas:")
print(df_so2.head())
print("\nTipos de datos:")
print(df_so2.dtypes)
print("\nPorcentaje de missings por columna:")
print((df_so2.isna().mean() * 100).sort_values(ascending=False))
print("\nNúmero de duplicados:")
print(df_so2.duplicated().sum())
print("\nNúmero de valores únicos por columna:")
print(df_so2.nunique())


EXPLORATORIO INICIAL — SO2

Shape (filas, columnas):
(1018390, 3)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 1018390 entries, 0 to 1018389
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   date           1018390 non-null  object 
 1   Samplingpoint  1018390 non-null  str    
 2   Value          1018390 non-null  float64
dtypes: float64(1), object(1), str(1)
memory usage: 41.8+ MB
None

Primeras filas:
         date        Samplingpoint     Value
0  2013-01-01  ES/SP_01022001_1_38  3.304348
1  2013-01-01  ES/SP_01036004_1_38  4.000000
2  2013-01-01  ES/SP_01051001_1_38  4.130435
3  2013-01-01  ES/SP_01059008_1_38  5.130435
4  2013-01-01  ES/SP_01059009_1_38  1.913043

Tipos de datos:
date              object
Samplingpoint        str
Value            float64
dtype: object

Porcentaje de missings por columna:
date             0.0
Samplingpoint    0.0
Value            0.0
dtype: float64

Número de duplicados:

In [ ]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE SO2

Shape (1.018.390, 3): dataset grande, SO2 se mide en bastantes estaciones aunque menos que NO2 u O3

Tipos: date como object, Samplingpoint como string y Value como float

Primeras filas: Valores entre 3-5 µg/m³, coherentes con niveles actuales de SO2 en España

Missings: 0% en todas las columnas

Duplicados: Ninguno

Valores únicos: 2556 fechas, 475 estaciones y 35.442 valores distintos
    dataset adecuado para análisis temporal y espacial
'''

In [22]:
name_co = "CO"

print("="*80)
print(f"EXPLORATORIO INICIAL — {name_co}")
print("="*80)

print("\nShape (filas, columnas):")
print(df_co.shape)
print("\nInfo:")
print(df_co.info())
print("\nPrimeras filas:")
print(df_co.head())
print("\nTipos de datos:")
print(df_co.dtypes)
print("\nPorcentaje de missings por columna:")
print((df_co.isna().mean() * 100).sort_values(ascending=False))
print("\nNúmero de duplicados:")
print(df_co.duplicated().sum())
print("\nNúmero de valores únicos por columna:")
print(df_co.nunique())


EXPLORATORIO INICIAL — CO

Shape (filas, columnas):
(508288, 3)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 508288 entries, 0 to 508287
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   date           508288 non-null  object 
 1   Samplingpoint  508288 non-null  str    
 2   Value          508288 non-null  float64
dtypes: float64(1), object(1), str(1)
memory usage: 20.8+ MB
None

Primeras filas:
         date        Samplingpoint     Value
0  2013-01-01  ES/SP_01022001_6_48  0.191739
1  2013-01-01  ES/SP_01055001_6_48  0.213043
2  2013-01-01  ES/SP_02003001_6_48  0.150957
3  2013-01-01  ES/SP_03009006_6_48  0.139130
4  2013-01-01  ES/SP_03014006_6_48  0.147826

Tipos de datos:
date              object
Samplingpoint        str
Value            float64
dtype: object

Porcentaje de missings por columna:
date             0.0
Samplingpoint    0.0
Value            0.0
dtype: float64

Número de duplicados:
0

Númer

In [ ]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE CO

Shape (508.288, 3): dataset más pequeño que los anteriores, lo cual es normal porque CO se mide en menos estaciones

Tipos: date como object, Samplingpoint como string y Value como float

Primeras filas: Valores entre 0.1-0.3 mg/m³, totalmente realistas para CO en España

Missings: 0% en todas las columnas

Duplicados: Ninguno

Valores únicos: 2.556 fechas, 285 estaciones y 48.817 valores distintos

'''

In [24]:
name_c6h6 = "C6H6"

print("="*80)
print(f"EXPLORATORIO INICIAL — {name_c6h6}")
print("="*80)

print("\nShape (filas, columnas):")
print(df_c6h6.shape)
print("\nInfo:")
print(df_c6h6.info())
print("\nPrimeras filas:")
print(df_c6h6.head())
print("\nTipos de datos:")
print(df_c6h6.dtypes)
print("\nPorcentaje de missings por columna:")
print((df_c6h6.isna().mean() * 100).sort_values(ascending=False))
print("\nNúmero de duplicados:")
print(df_c6h6.duplicated().sum())
print("\nNúmero de valores únicos por columna:")
print(df_c6h6.nunique())

EXPLORATORIO INICIAL — C6H6

Shape (filas, columnas):
(163708, 3)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 163708 entries, 0 to 163707
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   date           163708 non-null  object 
 1   Samplingpoint  163708 non-null  str    
 2   Value          163708 non-null  float64
dtypes: float64(1), object(1), str(1)
memory usage: 6.9+ MB
None

Primeras filas:
         date         Samplingpoint     Value
0  2013-01-01  ES/SP_03014006_30_59  0.608696
1  2013-01-01  ES/SP_04013004_30_59  1.956522
2  2013-01-01   ES/SP_04032005_30_I  0.420000
3  2013-01-01   ES/SP_04035002_30_I  0.440000
4  2013-01-01   ES/SP_04066004_30_I  0.670000

Tipos de datos:
date              object
Samplingpoint        str
Value            float64
dtype: object

Porcentaje de missings por columna:
date             0.0
Samplingpoint    0.0
Value            0.0
dtype: float64

Número de duplicados:
0

In [ ]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE C6H6

Shape (163.708, 3): es el dataset más pequeño, coherente con que el benceno se mide en muy pocas estaciones especializadas

Tipos: date como object, Samplingpoint como string y Value como float

Primeras filas: Valores entre 0.4-2 µg/m³, totalmente realistas para benceno

Missings: 0% en todas las columnas

Duplicados: ninguno

Valores únicos: 2.556 fechas, 131 estaciones y 17.901 valores distintos
'''

In [ ]:
"""
los contaminantes no requieren más limpieza porque el pipeline ya aplicó todas las transformaciones necesarias:
    carga por row groups ->evita problemas de memoria y lee solo columnas útiles
    conversión de tipos -> Start a datetime, Value a float, Samplingpoint a string
    filtrado temporal -> solo 2011-2019, consistente con los datasets de salud
    agregación diaria -> elimina duplicados, suaviza outliers horarios y homogeneiza el formato
    exploratorio final -> confirma 0% missings, 0 duplicados, valores realistas y estructura estable

el resultado es un esquema perfecto para análisis temporal, espacial y para unir con estaciones y datos de salud pública
"""

In [ ]:
# CIERRE DEL NOTEBOOK 02
'''
#
en este notebook he cargado los datasets originales de contaminantes, he aplicado una reducción eficiente
mediante carga por row groups, filtrado temporal al rango 2011-2019 y agregación diaria
el resultado son versiones limpias, homogéneas y manejables de cada contaminante, con las columnas
esenciales: date, Samplingpoint y Value

el exploratorio confirma que los datasets resultantes no contienen duplicados, no presentan valores nulos,
tienen tipos correctos y muestran rangos realistas, por tanto, no requieren más limpieza

en el siguiente notebook (03_context) se realizará el exploratorio y limpieza de los datasets de contexto
(hospitalizaciones, mortalidad, población y estaciones)
la integración completa de contaminantes + estaciones + contexto se llevará a cabo en el Notebook 04
'''